(mask_manager)=
# Mask Manager: General Masks and the Automatic Water Mask

The masking package turns a geographic extent into a ready-to-use validity
mask. It has two halves:

- the **general mask option** — a `RasterMask` (any single-band GeoTIFF, any
  CRS) or `VectorMask` (shapely/geopandas polygons), auto-resampled to the
  DEM grid and composed via `MaskOperator` (union / intersection / invert);
- the **automatic water mask** — a manager mirroring the PROPOSAL-0030
  `DEM.from_source`: it fetches source tiles through the public DEM API
  transport engine, extracts a boolean water layer, vectorizes it **once**
  into a shared cache, buffers it per-run (1 km default) in the ROI's
  auto-UTM zone, and rasterizes the result onto the DEM grid.

Masks report the removed region with the ISCE3 convention — value `1` =
water / removed, `0` = valid keep, `255` = invalid where no data exists —
and plug into the Stack as a *support* input (a `valid_mask` intersection
at IFG formation and the caller-supplied mask at unwrap).

This tutorial covers:

1. The selection grammar (`water` / `water:<provider>`) and its metadata
2. Building the cached vectorized water layer (default `water` = GSW
   occurrence)
3. Consuming the buffered binary mask onto the DEM grid
4. User-supplied masks: `RasterMask`, `VectorMask`, and `MaskOperator`
5. Stack usage: the `mask` configuration surface and `run(config)` keys
6. Coverage changes and fail-closed behavior
7. Provenance: the mask STAC asset and the vector by-product
8. Outage handling and environment variables

> **Prerequisites.** This notebook assumes you are running inside the
> project's `.venv` with all dependencies installed. `get_mask_manager`
> needs a cache folder; if `FANINSAR_MASK_CACHE_DIR` is already set in your
> environment it is reused, otherwise a temporary folder is created. The
> water sources are anonymous cloud channels — no account, no token, no API
> key — but the first fetch of a tile set does hit the network.

## Imports and cache setup


In [1]:
import json
import os
import tempfile
from pathlib import Path

from faninsar.processing.masking.mask import (
    MaskOperator,
    RasterMask,
    VectorMask,
    antimeridian_seam_guard,
    padded_fetch_band,
    snap_band,
)
from faninsar.processing.masking.mask_manager import (
    MaskProviderUnavailableError,
    get_mask_manager,
    resolve_auto_mask,
)
from faninsar.processing.masking.mask_sources import (
    get_mask_source,
    list_mask_sources,
    parse_mask_selection,
)
from faninsar.query import BoundingBox

cache_dir = Path(
    os.environ.get("FANINSAR_MASK_CACHE_DIR")
    or tempfile.mkdtemp(prefix="mask_cache_")
)
os.environ["FANINSAR_MASK_CACHE_DIR"] = str(cache_dir)
print("mask cache:", cache_dir)

manager = get_mask_manager()  # no argument -> FANINSAR_MASK_SOURCE -> water (GSW)
print("default selection:", manager.source_entry.name)

mask cache: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/mask_cache_s7emr_st
default selection: gsw


## 1. The selection grammar

`list_mask_sources()` returns every registered name. The selection grammar
is `"water"` or `"water:<provider>"`; `"water"` is the auto alias that
resolves to the wired default provider (GSW occurrence).


In [2]:
names = list_mask_sources()
print(f"{len(names)} selection names:")
print(", ".join(names))

# Resolve every name and show the two-axis metadata.
header = (
    f"{'selection':<12} {'provider':<13} {'resolution':>10} {'tile':>5}"
    f"  {'extraction':<11} {'water predicate':<20} auth"
)
print("\n" + header + "\n" + "-" * len(header))
for name in names:
    entry = get_mask_source(name)
    resolution = f"{entry.resolution_m:.0f} m" if entry.resolution_m else "-"
    tile = f"{entry.tile_size_deg:.0f} deg" if entry.tile_size_deg else "-"
    if entry.extraction == "threshold":
        predicate = f"occurrence >= {entry.threshold}"
    elif entry.extraction == "categorical":
        predicate = f"class in {sorted(entry.excluded_values)}"
    else:
        predicate = "already vector"
    label = f"{name} (auto)" if name == "water" else name
    print(
        f"{label:<12} {entry.provider:<13} {resolution:>10} {tile:>5}"
        f"  {entry.extraction:<11} {predicate:<20} {entry.auth}"
    )

try:
    parse_mask_selection("water:osm-overpass")  # registered but unwired
except Exception as err:
    print("\nosm-overpass rejected before any network traffic:", type(err).__name__)

3 selection names:
gsw, water, worldcover

selection    provider      resolution  tile  extraction  water predicate      auth
----------------------------------------------------------------------------------
gsw          gsw                 30 m 10 deg  threshold   occurrence >= 50     none
water (auto) gsw                 30 m 10 deg  threshold   occurrence >= 50     none
worldcover   worldcover          10 m 3 deg  categorical class in [80]        none

osm-overpass rejected before any network traffic: ValueError


Two columns deserve attention:

- **extraction** — `threshold` sources (GSW occurrence) binarize with
  `value >= threshold` (default 50); `categorical` sources (WorldCover)
  binarize by class membership (`excluded_values={80}`); `vector` sources
  arrive as polygons and skip extraction (OSM — not wired until the
  Overpass response-size cap and truncation detection land).
- **polarity** — in all cases the extracted boolean marks **water /
  removed** (`1`); `invert=True` flips the predicate and is applied last.
  NoData cells never count as water.

All v1 providers are anonymous (`auth: none`): GSW lives on Google Cloud
Storage, WorldCover on AWS S3. `water:osm-overpass` is registered but fails
closed until its response-size cap binds.

## 2. The automatic water mask: fetch, extract, vectorize, cache

`MaskManager.get_water_layer(bounds)` runs the pipeline: fetch the raw
tiles covering the **padded fetch band**, extract the boolean water,
vectorize once, and cache the GeoJSON layer with a provenance sidecar. The
padded fetch band is the ROI expanded by `buffer_km` in the ROI's auto-UTM
zone and snapped to the source tile grid, so buffered water from a
neighboring tile is never silently missing at ROI edges.


In [3]:
roi = BoundingBox(0.6, 5.5, 1.3, 5.95)  # Keta Lagoon coast, Ghana
bounds = (roi.left, roi.bottom, roi.right, roi.top)
buffer_km = 1.0

padded = padded_fetch_band(bounds, buffer_km, zone_lon=0.95, zone_lat=5.725)
band = snap_band(padded, 3.0)  # WorldCover tiles are 3-degree
print("ROI:", bounds)
print("padded fetch band:", tuple(round(v, 6) for v in padded))
print("tile-snapped band:", band)

wc = get_mask_manager(source="water:worldcover")
layer = wc.get_water_layer(padded)
print("\nvector layer:", layer.path.name)
print("features:", layer.feature_count, "| from_cache:", layer.from_cache)
print("identity:", layer.identity[:16] + "...",
      "| source_version:", layer.source_version[:16] + "...")

ROI: (0.6, 5.5, 1.3, 5.95)
padded fetch band: (0.590934, 5.490925, 1.30905, 5.95907)
tile-snapped band: (0.0, 3.0, 3.0, 6.0)

vector layer: layer.geojson
features: 10 | from_cache: False
identity: baad39fa498eb6da... | source_version: 3acaff82cf7fc458...


The cache location encodes identity:
`<cache_dir>/<product>-<provider>/vectors/<identity>/layer.geojson`, where
`identity` is a sha256 digest over `(source, tile-snapped padded bounds,
resolved plan URL/version, threshold, excluded_values, invert, simplify
tolerance, minimum area)`. Any extraction-parameter change **or
source-version bump** produces a new identity and re-extracts; identical
parameters reuse the cache across runs — a second call skips both the tile
fetch and the vectorization. Bounds are tile-snapped before hashing, so
float jitter cannot defeat cross-run reuse.

The fetch is **atomic at the set level**: any tile failure fails the whole
call with `MaskProviderUnavailableError`, and a partial tile set is never
vectorized into a mask (completed tiles stay cached and a retry resumes).


In [4]:
layer_again = wc.get_water_layer(padded)
print("second call from_cache:", layer_again.from_cache)
print("(cache hit: no tile download, no re-vectorization)")

print("\ncache layout under", wc.partition_name + ":")
for path in sorted(cache_dir.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(cache_dir)}  ({path.stat().st_size / 1e6:.1f} MB)")

second call from_cache: True
(cache hit: no tile download, no re-vectorization)

cache layout under water-worldcover:
  water-worldcover/ESA_WorldCover_10m_2021_v200_N03E000_Map.tif  (5.4 MB)
  water-worldcover/vectors/baad39fa498eb6da43a7ea264591fb7929f765e442cca2e09f6197c3f6e55ba0/layer.geojson  (0.8 MB)
  water-worldcover/vectors/baad39fa498eb6da43a7ea264591fb7929f765e442cca2e09f6197c3f6e55ba0/provenance.json  (0.0 MB)


## 3. Consume: buffered binary mask on the DEM grid

`resolve_auto_mask` (module-level convenience, or
`MaskManager.resolve_auto_mask`) loads the cached vector, buffers it
per-run with `buffer_land_utm_km` in the ROI's auto-UTM zone
(metric-correct — no cos(lat) degree conversion), and rasterizes onto the
DEM mosaic grid with `rasterize_to_grid`. Changing `buffer_km` never
re-downloads: the buffer is not part of the vector identity, and only the
small per-grid raster cache entry is regenerated.


In [5]:
import numpy as np
import rasterio
from rasterio.transform import from_bounds

out_dir = Path(tempfile.mkdtemp(prefix="mask_out_"))
out_dir_2km = Path(tempfile.mkdtemp(prefix="mask_out_2km_"))
dem_path = out_dir / "dem.tif"
res = 0.005  # ~500 m cells for a small tutorial DEM
width = int((roi.right - roi.left) / res)
height = int((roi.top - roi.bottom) / res)
dem_transform = from_bounds(roi.left, roi.bottom, roi.right, roi.top, width, height)
elevation = (np.add.outer(np.arange(height), np.arange(width)) % 97).astype("float32") * 10.0
elevation[:8, :8] = -9999.0  # a no-data hole -> mask value 255 there
with rasterio.open(
    dem_path, "w", driver="GTiff", height=height, width=width, count=1,
    dtype="float32", crs="EPSG:4326", transform=dem_transform, nodata=-9999.0,
) as dst:
    dst.write(elevation, 1)

mask_path = resolve_auto_mask(
    roi, dem_path=dem_path, output_dir=out_dir, buffer_km=1.0,
    on_failure="error", source="water:worldcover",
)
mask_path_2km = resolve_auto_mask(
    roi, dem_path=dem_path, output_dir=out_dir_2km, buffer_km=2.0,
    on_failure="error", source="water:worldcover",
)

with rasterio.open(mask_path) as src:
    plane = src.read(1)
    tags_1km = src.tags()
with rasterio.open(mask_path_2km) as src:
    plane_2km = src.read(1)
    tags_2km = src.tags()

values, counts = np.unique(plane, return_counts=True)
print("mask product:", mask_path)
print("uint8 values (1 km buffer):", dict(zip(values.tolist(), counts.tolist())))
print(
    "same vector-layer identity for both buffers:",
    tags_1km["mask_identity"] == tags_2km["mask_identity"],
)

mask product: /var/folders/4f/zhln6dbs3kxb5kpx5pclzwsc0000gn/T/mask_out_1qhzhvjw/mask/water_mask.tif
uint8 values (1 km buffer): {0: 2101, 1: 10435, 255: 64}
same vector-layer identity for both buffers: True


The product is a uint8 GeoTIFF on the DEM grid with the pinned conventions:

- `0` — valid keep (land beyond the buffer),
- `1` — water / removed (the buffered water polygon),
- `255` — invalid where the DEM itself has no data.

The rasterized product caches under
`<product>-<provider>/rasters/<identity>/` keyed by
`(vector-layer digest, buffer_km, grid + DEM data mask)`, so an unchanged
configuration never re-rasterizes on a later run. Note the `on_failure`
argument: `"error"` (used here) raises the structured
`MaskProviderUnavailableError` when the provider fails; Stack automation
passes `"warning"` instead.

## 4. User-supplied masks: `RasterMask`, `VectorMask`, `MaskOperator`

A `RasterMask` accepts any single-band GeoTIFF (any CRS; non-geographic CRS
are reprojected on the fly for point lookups). Its fill semantics are
pinned by PROPOSAL-0039: cells of the target grid outside the raster's
extent resolve to **valid** (the exclusion intent never silently deletes
data), NoData cells resolve to **invalid**, and resampling is
**nearest-neighbour only** (bilinear would invent fractional values at
coastlines).


In [6]:
user_mask_path = out_dir / "user_land_mask.tif"
grid = np.zeros((40, 40), dtype="uint8")
grid[10:20, 10:20] = 80  # a class code marking the removed region
grid[30:34, 5:9] = 255   # NoData cells
user_transform = from_bounds(6.0, 3.0, 7.0, 4.0, 40, 40)
with rasterio.open(
    user_mask_path, "w", driver="GTiff", height=40, width=40, count=1,
    dtype="uint8", crs="EPSG:4326", transform=user_transform, nodata=255,
) as dst:
    dst.write(grid, 1)

umask = RasterMask(user_mask_path, excluded_values=frozenset({80}))

def point(row, col):
    lon, lat = user_transform * (col + 0.5, row + 0.5)
    return lat, lon

probes = {
    "excluded class (80)": point(15, 15),
    "NoData (255)": point(32, 7),
    "plain land (0)": point(5, 5),
    "outside the raster extent": point(45, 5),
}
lats = np.array([lat for lat, _ in probes.values()])
lons = np.array([lon for _, lon in probes.values()])
keep = umask.sample(lats, lons)
print("RasterMask keep plane (True = keep):")
for (label, _), k in zip(probes.items(), keep.tolist()):
    print(f"  {label:<28} keep = {k}")

inverted = RasterMask(user_mask_path, excluded_values=frozenset({80}), invert=True)
keep_inv = inverted.sample(lats, lons)
print("\nWith invert=True (the predicate flips; fill rules hold):")
for (label, _), k in zip(probes.items(), keep_inv.tolist()):
    print(f"  {label:<28} keep = {k}")

RasterMask keep plane (True = keep):
  excluded class (80)          keep = False
  NoData (255)                 keep = False
  plain land (0)               keep = True
  outside the raster extent    keep = True

With invert=True (the predicate flips; fill rules hold):
  excluded class (80)          keep = True
  NoData (255)                 keep = False
  plain land (0)               keep = False
  outside the raster extent    keep = True


In [7]:
from shapely.geometry import Polygon

lake = Polygon([(6.60, 3.30), (6.90, 3.30), (6.90, 3.55), (6.60, 3.55)])
vmask = VectorMask([lake])

plats = np.array([3.45, 3.90])   # inside the lake, outside the lake
plons = np.array([6.75, 6.20])
print("VectorMask keep (True = outside every geometry):")
print("  inside the lake:", vmask.sample(plats, plons)[0])
print("  outside the lake:", vmask.sample(plats, plons)[1])

plane_lake = vmask.rasterize(user_transform, (40, 40))
print("rasterized removed pixels:", int((plane_lake == 1).sum()))

user_mask = RasterMask(user_mask_path, excluded_values=frozenset({80}))
both = MaskOperator.union(user_mask, vmask)  # removed = class-80 blocks ∪ lake
print("\nMaskOperator union: keep inside the lake =",
      both.sample(plats, plons)[0])
either_keep = MaskOperator.intersection(user_mask, vmask)  # removed ∩ removed
print("MaskOperator intersection: keep outside the lake =",
      either_keep.sample(plats, plons)[1])
complement = MaskOperator.invert(user_mask)  # kept <-> removed
lat80, lon80 = point(15, 15)
print("MaskOperator invert: keep inside the class-80 block =",
      complement.sample(np.array([lat80]), np.array([lon80]))[0])

VectorMask keep (True = outside every geometry):
  inside the lake: False
  outside the lake: True
rasterized removed pixels: 120

MaskOperator union: keep inside the lake = False
MaskOperator intersection: keep outside the lake = True
MaskOperator invert: keep inside the class-80 block = True


## 5. Stack usage

The Stack defaults to the automatic water mask. `mask=None` (or the
`"none"` spelling accepted from mapping configs) **explicitly disables
masking and restores unmasked processing** — the exact legacy behavior,
with no mask lineage recorded.


In [8]:
import dataclasses

from faninsar.processing.stack.config import (
    AUTO_WATER_MASK,
    MASK_DISABLED,
    StackConfig,
)

print("AUTO_WATER_MASK:", repr(AUTO_WATER_MASK))
print("explicit-disable spelling:", repr(MASK_DISABLED))
print()
for f in dataclasses.fields(StackConfig):
    if f.name in {
        "mask",
        "mask_source",
        "mask_resolution_m",
        "mask_buffer_km",
        "ocean_shore_keep_m",
        "inland_shore_keep_m",
        "mask_on_failure",
        "mask_apply_ionosphere",
    }:
        print(f"{f.name:<26} default = {f.default!r}")

cfg = StackConfig(work_dir=out_dir, activation_mode="reference", mask="none")
print("\nStackConfig(mask='none') normalizes to:", cfg.mask)

AUTO_WATER_MASK: 'water'
explicit-disable spelling: 'none'

mask                       default = 'water'
mask_source                default = None
mask_resolution_m          default = None
mask_buffer_km             default = 1.0
ocean_shore_keep_m         default = 1000
inland_shore_keep_m        default = 0
mask_on_failure            default = 'warning'
mask_apply_ionosphere      default = False

StackConfig(mask='none') normalizes to: None


`run(config)` accepts the matching keys. `mask: none` is the
explicit-disable spelling for mapping configs; an absent key keeps the
auto-water default.

```yaml
paths: [s1a_20230607.zip, s1a_20230701.zip]
output: /data/stacks/gulf-of-guinea
roi: [0.6, 5.5, 1.3, 5.95]
mask: water              # default; 'none' restores unmasked processing
mask_source: water:gsw   # optional selection override (water / water:<provider>)
mask_buffer_km: 1.0      # land buffer extended into water
mask_resolution_m: null  # null -> rasterize onto the DEM mosaic grid
mask_on_failure: warning # Stack default: degrade to unmasked, never block
mask_apply_ionosphere: false
```

The resolved mask is applied as a `valid_mask` intersection at IFG
formation (`valid_mask &= ~mask`), handed to the spatial unwrappers as the
caller-supplied mask, and applied to the geocoded SLC validity — but
**not** to ionosphere estimation unless `mask_apply_ionosphere: true`. When
a `warning`/`skip` run degrades to unmasked, the run manifest records
`{"mask": "absent", "reason": ...}` so the degraded state stays
machine-checkable; a resolved mask records `{"mask": "present", ...}` with
provenance.

## 6. Coverage changes and fail-closed behavior

An active water mask **shrinks processed coverage**: at burst-selection
level the Stack computes `effective_ROI = ROI − water_beyond_buffer`, so
bursts and pixels over open water (plus the `mask_buffer_km` land strip)
are never processed. This is intended, but it changes output coverage —
disable the mask with `mask=None` when you need the unmasked footprint
back.

Two structural guards hold **regardless of `mask_on_failure`** (including
the Stack default `warning`):

- **antimeridian seam band** — ROIs whose padded fetch band reaches ±180°
  are rejected fail-closed before any fetch (buffered disks would wrap the
  seam and crash intermittently). The guard evaluates three conditions on
  the raw bounds: longitudes outside [-180, 180], a planar longitude span
  greater than 180°, and the tile-snapped padded band touching ±180°.
- **empty effective ROI** — a ROI fully covered by buffered water
  (all-water ROI, misdrawn ROI, or an extraction predicate that marks
  everything as water) raises a structured `InvalidProcessingStateError`
  naming the ROI and mask configuration, instead of silently selecting
  zero bursts.


In [9]:
def seam_report(label, raw, tile_size_deg):
    zone_lon = (raw[0] + raw[2]) / 2.0
    zone_lat = (raw[1] + raw[3]) / 2.0
    band = snap_band(
        padded_fetch_band(raw, 1.0, zone_lon=zone_lon, zone_lat=zone_lat),
        tile_size_deg,
    )
    ok, reason = antimeridian_seam_guard(raw, padded_band=band)
    print(f"{label:<42} {'pass' if ok else 'REJECTED'}: {reason}")

seam_report("inland ROI (99.5E..100.5E, 38.5N..39.5N)", (99.5, 38.5, 100.5, 39.5), 10.0)
seam_report("exact-180-degree hemisphere ROI", (-90.0, -45.0, 90.0, 45.0), 10.0)
seam_report("wrap evader (-170..170 drawn across the seam)", (-170.0, 8.0, 170.0, 9.0), 10.0)
seam_report("unwrapped longitudes (175..185)", (175.0, 8.0, 185.0, 9.0), 10.0)
seam_report("near-seam ROI (179.995E + 1 km buffer)", (179.5, 8.0, 179.995, 9.0), 3.0)

inland ROI (99.5E..100.5E, 38.5N..39.5N)   pass: ok
exact-180-degree hemisphere ROI            pass: ok
wrap evader (-170..170 drawn across the seam) REJECTED: planar longitude span > 180 deg (seam wrap)
unwrapped longitudes (175..185)            REJECTED: out-of-range longitude (outside [-180, 180])
near-seam ROI (179.995E + 1 km buffer)     REJECTED: padded fetch band reaches the +/-180 seam


## 7. Provenance and products

Every mask artifact carries provenance. The buffered binary mask is written
under `<output_dir>/mask/` with GeoTIFF tags, and in a Stack run it
registers beside the pair Zarr store as the STAC `mask` asset
(`1 = water/removed`, `image/tiff`, roles `["mask", "data"]`) with the
provenance keys surfaced as `faninsar:mask_*` item properties. The
vectorized water layer is a **durable by-product of the shared mask
cache** — reused across runs and ROIs, never regenerated per run.


In [10]:
print("GeoTIFF provenance tags of", mask_path.name + ":")
for key in sorted(tags_1km):
    if key != "AREA_OR_POINT":
        print(f"  {key:<18} = {tags_1km[key]}")

provenance_path = wc.partition_dir / "vectors" / layer.identity / "provenance.json"
print("\nvector-layer provenance sidecar:", provenance_path.relative_to(cache_dir))
print(json.dumps(json.loads(provenance_path.read_text(encoding="utf-8")), indent=1))

GeoTIFF provenance tags of water_mask.tif:
  excluded_values    = 80
  invert             = false
  mask_buffer_km     = 1.0
  mask_identity      = baad39fa498eb6da43a7ea264591fb7929f765e442cca2e09f6197c3f6e55ba0
  mask_product       = water
  mask_provider      = worldcover
  mask_retrieved     = 2026-08-30
  source_version     = 3acaff82cf7fc4582a7fd575426d50d3

vector-layer provenance sidecar: water-worldcover/vectors/baad39fa498eb6da43a7ea264591fb7929f765e442cca2e09f6197c3f6e55ba0/provenance.json
{
 "mask_product": "water",
 "mask_provider": "worldcover",
 "mask_retrieved": "2026-08-30",
 "threshold": null,
 "excluded_values": [
  80
 ],
 "invert": false,
 "source_version": "3acaff82cf7fc4582a7fd575426d50d3"
}


## 8. Outage handling and environment variables

A failing provider raises `MaskProviderUnavailableError` — no hidden
cross-provider fallback, no partial mask. The exception carries structured
fields so callers can decide what to do next:


In [11]:
err = MaskProviderUnavailableError(
    "storage.googleapis.com is unreachable for water",
    product="water",
    provider="gsw",
    host="storage.googleapis.com",
    failure_class="upstream-outage",
    attempts=3,
)
print(
    f"{err.failure_class} on {err.product}@{err.provider} ({err.host})"
    f" after {err.attempts} attempt(s)"
)
print("failure classes: auth, forbidden, coverage, upstream-outage")

upstream-outage on water@gsw (storage.googleapis.com) after 3 attempt(s)
failure classes: auth, forbidden, coverage, upstream-outage


The `on_failure` policy decides what happens next:

| Policy | Behavior |
|---|---|
| `error` | raise `MaskProviderUnavailableError` (class default; explicit user masks) |
| `warning` | log loudly, continue **unmasked**, record `{"mask": "absent", "reason": ...}` in the run manifest (Stack default) |
| `skip` | continue silently |

Environment variables (an explicit argument always wins over the
environment; the environment always wins over the built-in default):

| Variable | Meaning | Default |
|---|---|---|
| `FANINSAR_MASK_CACHE_DIR` | cache root for raw tiles, vector layers, and rasterized products (required) | — |
| `FANINSAR_MASK_SOURCE` | selection: `water` / `water:<provider>` | `water` |
| `FANINSAR_MASK_SOURCE_URL` | https base-URL override (mirror escape hatch); URLs embedding `user:pass@host` credentials are rejected fail-closed at runtime | provider default |

Cache locations under `FANINSAR_MASK_CACHE_DIR`:

| Path | Content |
|---|---|
| `<product>-<provider>/` | raw source tiles |
| `<product>-<provider>/vectors/<identity>/` | cached GeoJSON layer + `provenance.json` |
| `<product>-<provider>/rasters/<identity>/` | cached buffered binary masks keyed by grid + buffer |

## Exercise

Re-resolve the tutorial mask with a third buffer width and answer:

1. How does the removed fraction change between the 1 km and 2 km buffers?
2. Was any tile re-downloaded for the 2 km resolution? (Compare the
   `mask_identity` tags of the two products.)
3. Which cache directory holds the rasterized products?


In [12]:
print(f"removed fraction, 1 km buffer: {(plane == 1).mean():.1%}")
print(f"removed fraction, 2 km buffer: {(plane_2km == 1).mean():.1%}")

raster_dir = wc.partition_dir / "rasters" / layer.identity
print("cached raster products:", len(list(raster_dir.glob("*.tif"))))

removed fraction, 1 km buffer: 82.8%
removed fraction, 2 km buffer: 86.6%
cached raster products: 2


## Pitfalls and extensions

- **Bounds order.** All bounds are `(min_lon, min_lat, max_lon, max_lat)`
  in EPSG:4326 — longitude first.
- **Coverage shrinkage is intended.** `ROI − water_beyond_buffer` removes
  open water and the buffered land strip from burst selection; use
  `mask=None` to restore the unmasked footprint.
- **No silent failover.** A dead provider raises
  `MaskProviderUnavailableError`; the `on_failure` policy owns the
  loudness, and a `warning` run is recorded as mask-absent in the run
  manifest.
- **Structural guards ignore `on_failure`.** The antimeridian seam band and
  the empty-effective-ROI checks raise regardless of the policy.
- **Boolean masks only.** Continuous rasters are binarized via `threshold`
  (e.g. GSW occurrence >= 50); nearest-neighbour is the only mask
  resampling.
- **Shore policies.** `ocean_shore_keep_m=1000` / `inland_shore_keep_m=0`
  be configured together; equal values collapse to a single land buffer,
  and differing values fail closed until water-body connectivity
  classification is wired.
- **Mirror override.** `base_url=` (or `FANINSAR_MASK_SOURCE_URL`) points
  the selected source at a different https mirror; userinfo credentials in
  the mirror URL are rejected fail-closed, and pinned-endpoint sources
  refuse the override.
- **Manager knobs.** `MaskManager(cache_dir, ..., threshold=30,
  min_area_km2=0.1)` retunes extraction; any change produces a new vector
  identity and re-extracts once.